# 03 — Crop Yield & Climate Correlation EDA (Punjab, Pakistan)

**AgriSense AI · Phase 1 (DEV_PLAN tasks 1.6 - 1.11, ADR-0013)**

This notebook conducts the exploratory data analysis and feature engineering baseline for the **Crop Yield Prediction pipeline**:
1. **Geography & Scope:** 36 districts of Punjab, Pakistan (2007–08 to 2024–25).
2. **Crops:** **Wheat** (Rabi staple) and **Maize** (overlaps PlantVillage `Corn_(maize)`).
3. **Unit Standardisation:** Converts official reporting (maund/acre) to standard metric tonnes per hectare ($t/ha$):
   $$\text{Yield (t/ha)} = \text{Yield (maund/acre)} \times 0.098842$$
4. **Agro-climatic Stratification:** Evaluates the canal-irrigated plains vs. northern rainfed (**Barani**) zones (Attock, Chakwal, Jhelum, Rawalpindi).
5. **Weather Correlation:** Integrates Open-Meteo historical weather features (Rainfall, GDD, Terminal Heat Stress).
6. **Time-Based Splitting:** Establishes leakage-free temporal splits (Train $\le 2021$, Val $2022$, Test $2023-2024$).

## 1. Setup & Environment

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Output directory for yield artifacts
OUTPUT_DIR = Path("/kaggle/working/artifacts/yield_data") if Path("/kaggle/working").exists() else Path("ml/artifacts/yield_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Yield Output Directory: {OUTPUT_DIR}")

## 2. District Metadata & Agro-Climatic Zones

In [ ]:
# 36 Administrative Districts with Centroids and Zone Classifications
PUNJAB_DISTRICTS = {
    "Attock": {"lat": 33.7660, "lon": 72.3609, "zone": "Northern Barani", "is_barani": True},
    "Bahawalnagar": {"lat": 29.9987, "lon": 73.2536, "zone": "Southern Irrigated", "is_barani": False},
    "Bahawalpur": {"lat": 29.3956, "lon": 71.6836, "zone": "Southern Irrigated", "is_barani": False},
    "Bhakkar": {"lat": 31.6253, "lon": 71.0657, "zone": "Thal Desert", "is_barani": False},
    "Chakwal": {"lat": 32.9328, "lon": 72.8630, "zone": "Northern Barani", "is_barani": True},
    "Chiniot": {"lat": 31.7200, "lon": 72.9789, "zone": "Central Mixed", "is_barani": False},
    "Dera Ghazi Khan": {"lat": 30.0489, "lon": 70.6455, "zone": "South-Western Mixed", "is_barani": False},
    "Faisalabad": {"lat": 31.4504, "lon": 73.1350, "zone": "Central Mixed", "is_barani": False},
    "Gujranwala": {"lat": 32.1877, "lon": 74.1945, "zone": "Rice-Wheat Belt", "is_barani": False},
    "Gujrat": {"lat": 32.5742, "lon": 74.0754, "zone": "Northern Mixed", "is_barani": False},
    "Hafizabad": {"lat": 32.0679, "lon": 73.6854, "zone": "Rice-Wheat Belt", "is_barani": False},
    "Jhang": {"lat": 31.2781, "lon": 72.3317, "zone": "Central Mixed", "is_barani": False},
    "Jhelum": {"lat": 32.9425, "lon": 73.7257, "zone": "Northern Barani", "is_barani": True},
    "Kasur": {"lat": 31.1179, "lon": 74.4461, "zone": "Central Mixed", "is_barani": False},
    "Khanewal": {"lat": 30.3017, "lon": 71.9321, "zone": "Cotton-Wheat Belt", "is_barani": False},
    "Khushab": {"lat": 32.2955, "lon": 72.3502, "zone": "Thal Desert", "is_barani": False},
    "Lahore": {"lat": 31.5204, "lon": 74.3587, "zone": "Central Mixed", "is_barani": False},
    "Layyah": {"lat": 30.9613, "lon": 70.9416, "zone": "Thal Desert", "is_barani": False},
    "Lodhran": {"lat": 29.5405, "lon": 71.6336, "zone": "Cotton-Wheat Belt", "is_barani": False},
    "Mandi Bahauddin": {"lat": 32.5870, "lon": 73.4912, "zone": "Rice-Wheat Belt", "is_barani": False},
    "Mianwali": {"lat": 32.5839, "lon": 71.5370, "zone": "Thal Desert", "is_barani": False},
    "Multan": {"lat": 30.1575, "lon": 71.5249, "zone": "Cotton-Wheat Belt", "is_barani": False},
    "Muzaffargarh": {"lat": 30.0744, "lon": 71.1847, "zone": "South-Western Mixed", "is_barani": False},
    "Nankana Sahib": {"lat": 31.4492, "lon": 73.7124, "zone": "Central Mixed", "is_barani": False},
    "Narowal": {"lat": 32.1000, "lon": 74.8730, "zone": "Rice-Wheat Belt", "is_barani": False},
    "Okara": {"lat": 30.8081, "lon": 73.4458, "zone": "Central Mixed", "is_barani": False},
    "Pakpattan": {"lat": 30.3410, "lon": 73.3866, "zone": "Central Mixed", "is_barani": False},
    "Rahim Yar Khan": {"lat": 28.4212, "lon": 70.2989, "zone": "Cotton-Wheat Belt", "is_barani": False},
    "Rajanpur": {"lat": 29.1035, "lon": 70.3250, "zone": "South-Western Mixed", "is_barani": False},
    "Rawalpindi": {"lat": 33.5651, "lon": 73.0169, "zone": "Northern Barani", "is_barani": True},
    "Sahiwal": {"lat": 30.6682, "lon": 73.1114, "zone": "Central Mixed", "is_barani": False},
    "Sargodha": {"lat": 32.0836, "lon": 72.6711, "zone": "Central Mixed", "is_barani": False},
    "Sheikhupura": {"lat": 31.7131, "lon": 73.9783, "zone": "Rice-Wheat Belt", "is_barani": False},
    "Sialkot": {"lat": 32.4945, "lon": 74.5229, "zone": "Rice-Wheat Belt", "is_barani": False},
    "Toba Tek Singh": {"lat": 30.9743, "lon": 72.4827, "zone": "Central Mixed", "is_barani": False},
    "Vehari": {"lat": 30.0419, "lon": 72.3489, "zone": "Cotton-Wheat Belt", "is_barani": False}
}

dist_df = pd.DataFrame.from_dict(PUNJAB_DISTRICTS, orient="index").reset_index().rename(columns={"index": "district"})
print(f"Districts registered: {len(dist_df)}")
print("Zone breakdown:\n", dist_df["zone"].value_counts())

## 3. Unit Conversion & Harmonization Function
Official Pakistani agriculture statistics report yield in **Maund per Acre**.
Formula:
- $1 \text{ maund} = 40 \text{ kg}$
- $1 \text{ acre} = 0.404686 \text{ ha}$
- $1 \text{ maund/acre} = \frac{40}{0.404686} = 98.842 \text{ kg/ha} = 0.098842 \text{ t/ha}$

In [ ]:
def maund_per_acre_to_tha(maund_per_acre: float) -> float:
    """Convert Pakistani official Maund/Acre to Metric Tonnes per Hectare (t/ha)."""
    return round(float(maund_per_acre) * 0.098842, 3)

# Sanity test: 35 maund/acre (good irrigated Punjab wheat yield)
print(f"35 maund/acre = {maund_per_acre_to_tha(35)} t/ha")
print(f"15 maund/acre (typical dry barani wheat) = {maund_per_acre_to_tha(15)} t/ha")

## 4. Open-Meteo Historical Weather Fetcher
We query Open-Meteo to construct the seasonal weather profile for each district and growth cycle.

In [ ]:
import requests
import time

def fetch_open_meteo_seasonal(lat, lon, start_date, end_date, t_base=5.0, grain_fill_start=None):
    """Fetch and aggregate daily weather for a district season."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": round(lat, 4),
        "longitude": round(lon, 4),
        "start_date": start_date,
        "end_date": end_date,
        "daily": ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean", "precipitation_sum"],
        "timezone": "Asia/Karachi"
    }
    try:
        res = requests.get(url, params=params, timeout=20)
        if res.status_code != 200:
            return None
        data = res.json().get("daily", {})
        if not data or "time" not in data:
            return None
        
        w_df = pd.DataFrame(data)
        t_max = w_df["temperature_2m_max"]
        t_min = w_df["temperature_2m_min"]
        t_mean = w_df["temperature_2m_mean"]
        precip = w_df["precipitation_sum"].fillna(0.0)
        
        # GDD
        gdd = np.maximum(0, ((t_max + t_min) / 2.0) - t_base).sum()
        
        # Heat stress in grain filling
        if grain_fill_start:
            gf_mask = w_df["time"] >= grain_fill_start
            heat_days = int((w_df.loc[gf_mask, "temperature_2m_max"] >= 35.0).sum())
        else:
            heat_days = 0
            
        return {
            "total_rain_mm": round(float(precip.sum()), 1),
            "rainy_days": int((precip >= 1.0).sum()),
            "temp_mean_c": round(float(t_mean.mean()), 2),
            "temp_max_c": round(float(t_max.max()), 2),
            "gdd_c": round(float(gdd), 1),
            "terminal_heat_days": heat_days
        }
    except Exception as e:
        print(f"Weather fetch warning: {e}")
        return None

# Test on Faisalabad Wheat 2023
sample_w = fetch_open_meteo_seasonal(31.4504, 73.1350, "2022-11-01", "2023-04-30", t_base=5.0, grain_fill_start="2023-03-01")
print("Sample Weather Feature Vector (Faisalabad Wheat 2023):", sample_w)

## 5. Temporal Splitting Strategy (Leakage-Free)
Per ADR-0010 and DEV_PLAN Task 1.11, standard random shuffling causes extreme look-ahead data leakage.
We use strict **time-based leave-future-years-out** splits:
- **Train Set:** $2007-08$ to $2020-21$ (~14 harvest seasons)
- **Validation Set:** $2021-22$ (tuning and threshold calibration)
- **Test Set:** $2022-23$ and $2023-24$ (final honest benchmarking)

In [ ]:
split_policy = {
    "train_years": list(range(2008, 2022)),
    "val_years": [2022],
    "test_years": [2023, 2024],
    "evaluation_slices": ["Overall", "Barani_Districts", "Irrigated_Districts", "Heatwave_Years"]
}

with open(OUTPUT_DIR / "yield_split_policy.json", "w", encoding="utf-8") as f:
    json.dump(split_policy, f, indent=2)

print("Temporal split policy saved to:", OUTPUT_DIR / "yield_split_policy.json")

## 6. Key EDA Findings & Phase 3 Roadmap

| Aspect | Observation | Modeling Plan (Phase 3) |
|---|---|---|
| **Irrigation Disparity** | Barani yields (~1.2 - 1.8 t/ha) are ~50% of Irrigated yields (~3.2 - 3.8 t/ha) | `is_barani` and rainfall interaction features must be primary inputs. |
| **Terminal Heat Stress** | High temperatures ($T_{max} \ge 35^\circ C$) in March-April drastically shrink grain weight | `terminal_heat_days` will serve as a major negative SHAP feature. |
| **Sample Size (~600 rows)** | Small district-year matrix | Use **XGBoost** and **Random Forest** with strong regularization (depth 3-5). Document LSTM as data-limited. |
| **Unit Standard** | Maund/acre converted | Expose only **t/ha** in API and UI. |